In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import json
import hashlib
from typing import Literal, TypedDict, List, Annotated
import operator

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, MessagesState, END, START
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver



In [2]:
# ==========================================
# 1. IMPORT ALL TOOL MODULES FROM YOUR IMAGE


In [3]:
# ==========================================
from tools import (
    api_579_ffs_tool,
    asme_calculator,
    audit_trail,
    compliance_auditor,
    doc_generator,
    docker_sandbox,
    file_io,
    inspection_extractor_tool,
    material_lookup_tool,
    network_verifier,
    risk_based_inspection_tool,
    routing_guard,
    thickness_grid_analyzer,
)



In [4]:
# ==========================================
# 2. DEFINE SYSTEM STATE


In [5]:
from pydantic import BaseModel, Field
from typing import Annotated, Sequence
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages

class AgentState(BaseModel):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    retry_count: int = Field(default=0)
    max_retries: int = Field(default=3)
    human_approved: bool = Field(default=False)
    human_feedback: str = Field(default="")


In [6]:
# ==========================================
# 3. INITIALIZE OLLAMA LOCAL MODELS


In [7]:
# ==========================================
# Make sure these models are pulled locally in Ollama prior to execution:
#   ollama pull qwen2.5:3b
#   ollama pull qwen2.5-coder:1.5b
#   ollama pull deepseek-r1:1.5b
#   ollama pull llama3.2-vision (or similar local vision model)
#   ollama pull llama3.2:3b

supervisor_llm = ChatOllama(model="llama3.1:8b", temperature=0, format="json")
coder_llm = ChatOllama(model="llama3.1:8b", temperature=0, format="json")
reasoning_llm = ChatOllama(model="deepseek-r1:8b", temperature=0)
vision_llm = ChatOllama(model="llama3.1:8b", temperature=0, format="json")
reviewer_llm = ChatOllama(model="llama3.2:3b", temperature=0)



In [8]:
# ==========================================
# 4. ASSIGN TOOLS TO SPECIALIZED AGENTS


In [9]:
# Vision Agent Tools
vision_tools = [
    inspection_extractor_tool.extract_inspection_data,
    thickness_grid_analyzer.analyze_thickness_grid,
    file_io.read_scanned_pdf,
]

# Coder Agent Tools
coder_tools = [
    asme_calculator.calculate_asme_stresses,
    api_579_ffs_tool.run_ffs_assessment,
    material_lookup_tool.lookup_material,
    docker_sandbox.execute_in_sandbox,
]

# Reasoning / Compliance Agent Tools
reasoning_tools = [
    compliance_auditor.audit_cvc_compliance,
    risk_based_inspection_tool.calculate_rbi_score,
    routing_guard.verify_routing_policy,
]

# Publisher Agent Tools
publisher_tools = [
    doc_generator.generate_nfa_documents,
    audit_trail.write_sha256_audit_seal,
    network_verifier.verify_zero_egress,
]


In [10]:
# ==========================================
# 5. AGENT HELPER / WORKER AGENTS


In [11]:
# ==========================================
def get_next_node(last_message: BaseMessage, default_goto: str, state: AgentState) -> str:
    """Helper to detect retry thresholds or completion triggers."""
    if getattr(state, "retry_count", 0) >= getattr(state, "max_retries", 3):
        return "human_approval_gate"
    if "FINAL ANSWER" in last_message.content or "COMPLETED" in last_message.content:
        return "chief_reviewer"
    return default_goto

# Vision Worker Agent
vision_agent = create_react_agent(
    vision_llm,
    tools=vision_tools,
    prompt="You are a Vision & Extraction Agent. Extract parameters from scanned PDFs, inspection sheets, and thickness grids."
)

def vision_node(state: AgentState) -> Command[Literal["supervisor", "human_approval_gate"]]:
    result = vision_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "supervisor", state)
    result["messages"][-1] = HumanMessage(content=result["messages"][-1].content, name="vision_agent")
    return Command(update={"messages": result["messages"]}, goto=goto)

# Coder Worker Agent
coder_agent = create_react_agent(
    coder_llm,
    tools=coder_tools,
    prompt="You are a Senior Coder Agent. When using execute_in_sandbox, provide ONLY the python code as the code_string argument."
)

def coder_node(state: AgentState) -> Command[Literal["supervisor", "human_approval_gate"]]:
    result = coder_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "supervisor", state)
    result["messages"][-1] = HumanMessage(content=result["messages"][-1].content, name="coder_agent")
    return Command(update={"messages": result["messages"]}, goto=goto)

# Reasoning Worker Agent
reasoning_agent = create_react_agent(
    reasoning_llm,
    tools=reasoning_tools,
    prompt="You are a Compliance & Reasoning Agent. Perform CVC audits, RBI scores, and compliance evaluations."
)

def reasoning_node(state: AgentState) -> Command[Literal["supervisor", "human_approval_gate"]]:
    result = reasoning_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "supervisor", state)
    result["messages"][-1] = HumanMessage(content=result["messages"][-1].content, name="reasoning_agent")
    return Command(update={"messages": result["messages"]}, goto=goto)


# --- Adaptive Complexity Router ---
def route_question(state: AgentState) -> str:
    question = state.messages[-1].content
    system_prompt = (
        "You are a routing agent. Route the user's input.\n"
        "If it is a basic greeting (e.g. 'hi', 'hello') or a simple question that requires no external tools, output {\"destination\": \"direct_answer\"}.\n"
        "If it is a request to process a dossier, run calculations, extract data, or generate a report, output {\"destination\": \"supervisor\"}.\n"
        "CRITICAL: Output ONLY valid JSON with the key 'destination'."
    )
    import json
    response = supervisor_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    try:
        parsed = json.loads(response.content)
        return parsed.get("destination", "supervisor")
    except:
        return "supervisor"

def direct_answer_node(state: AgentState) -> dict:
    answer = supervisor_llm.invoke(state.messages).content
    return {"messages": [AIMessage(content=answer, name="direct_answer")]}
# ----------------------------------

# Supervisor Agent
def supervisor_node(state: AgentState) -> Command[Literal["vision_agent", "coder_agent", "reasoning_agent", "chief_reviewer", "human_approval_gate"]]:
    import re
    import json
    system_prompt = (
        "You are the Chief Supervisor Agent coordinating an engineering dossier processing system.\n"
        "Analyze the context and route tasks to one of these agents:\n"
        "- 'vision_agent': for extracted parameters, drawings, or scanned PDFs.\n"
        "- 'coder_agent': for ASME math, calculations, material lookups, or sandbox runs.\n"
        "- 'reasoning_agent': for compliance audits or RBI assessments.\n"
        "- 'chief_reviewer': if all required extractions, calculations, and compliance steps are complete.\n"
        "Respond with ONLY JSON format: {\"next\": \"<agent_name>\", \"instruction\": \"<task>\"}\n"
        "Do not include any other conversational text or markdown blocks."
    )
    messages = [SystemMessage(content=system_prompt)] + state.messages
    response = supervisor_llm.invoke(messages)
    
    try:
        # Use regex to find the first JSON object in the response
        match = re.search(r'\{.*?\}', response.content, re.DOTALL)
        if match:
            parsed = json.loads(match.group(0))
            next_target = parsed.get("next", "chief_reviewer")
        else:
            # Fallback if no {} found
            parsed = json.loads(response.content)
            next_target = parsed.get("next", "chief_reviewer")
    except Exception as e:
        print(f"Failed to parse Supervisor JSON: {e}")
        next_target = "chief_reviewer"

    if getattr(state, "retry_count", 0) >= getattr(state, "max_retries", 3):
        next_target = "human_approval_gate"

    return Command(
        update={"messages": [HumanMessage(content=response.content, name="supervisor")]},
        goto=next_target
    )

# Human Approval Gate Node
def human_approval_gate(state: AgentState) -> Command[Literal["supervisor", "chief_reviewer"]]:
    # Interrupt execution and wait for human engineer sign-off
    user_input = interrupt({
        "message": "CRITICAL: Max retries exceeded or safety flag raised. Human engineer sign-off required.",
        "status": "Awaiting Sign-off"
    })
    
    approved = user_input.get("approved", False)
    feedback = user_input.get("feedback", "")
    
    if approved:
        return Command(
            update={
                "human_approved": True,
                "human_feedback": feedback,
                "retry_count": 0,
                "messages": [HumanMessage(content=f"Human Engineer Approved: {feedback}", name="HumanGate")]
            },
            goto="chief_reviewer"
        )
    else:
        return Command(
            update={
                "human_approved": False,
                "human_feedback": feedback,
                "retry_count": 0,
                "messages": [HumanMessage(content=f"Human Engineer Rejected/Re-routed: {feedback}", name="HumanGate")]
            },
            goto="supervisor"
        )

# Chief Reviewer Agent
chief_reviewer_agent = create_react_agent(
    reviewer_llm,
    tools=[],
    prompt="You are the Chief Reviewer. Perform final safety, compliance, and sanity checks on all worker agent outputs. If there are critical flaws, say 'REJECT'. Otherwise say 'APPROVED'."
)

def chief_reviewer_node(state: AgentState) -> Command[Literal["deliverable_publisher"]]:
    result = chief_reviewer_agent.invoke(state)
    result["messages"][-1] = HumanMessage(content=result["messages"][-1].content, name="chief_reviewer")
    return Command(
        update={"messages": result["messages"]},
        goto="deliverable_publisher"
    )

# Deliverable Publisher Agent
publisher_agent = create_react_agent(
    supervisor_llm,
    tools=publisher_tools,
    prompt="You are the Deliverable Publisher. Compile outputs into Word/PDF documents and apply the SHA-256 audit seal."
)

def deliverable_publisher_node(state: AgentState) -> Command[Literal[END]]:
    result = publisher_agent.invoke(state)
    return Command(
        update={"messages": result["messages"]},
        goto=END
    )


C:\Users\DELL\AppData\Local\Temp\ipykernel_4648\4172981664.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  vision_agent = create_react_agent(
C:\Users\DELL\AppData\Local\Temp\ipykernel_4648\4172981664.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  coder_agent = create_react_agent(
C:\Users\DELL\AppData\Local\Temp\ipykernel_4648\4172981664.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  reasoning_agent = create_react_agent(
C:\Users\DELL\AppData\Local\Temp\ipykernel_4648\41729

In [12]:
# ==========================================
# 6. CONSTRUCT STATE GRAPH


In [13]:
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("direct_answer", direct_answer_node)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("vision_agent", vision_node)
workflow.add_node("coder_agent", coder_node)
workflow.add_node("reasoning_agent", reasoning_node)
workflow.add_node("human_approval_gate", human_approval_gate)
workflow.add_node("chief_reviewer", chief_reviewer_node)
workflow.add_node("deliverable_publisher", deliverable_publisher_node)

# Set Graph Entry Point (Adaptive Router)
workflow.add_conditional_edges(START, route_question, {
    "direct_answer": "direct_answer",
    "supervisor": "supervisor"
})
workflow.add_edge("direct_answer", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


In [14]:
# ==========================================
# 7. EXECUTION EXAMPLE


In [15]:
# ==========================================
if __name__ == "__main__":
    initial_input = {
        "messages": [
            HumanMessage(
                content="Process dossier at data/test_dossiers/ID-9982_UT_Scan.pdf. Extract thickness grid, run ASME Section VIII Div 1 calculations, check CVC compliance, and publish signed NFA."
            )
        ],
        "retry_count": 0,
        "max_retries": 3,
        "human_approved": False,
        "human_feedback": ""
    }

    # Stream execution through Ollama LLM nodes
    for chunk in app.stream(initial_input, stream_mode="values"):
        latest_msg = chunk["messages"][-1]
        print(f"[{getattr(latest_msg, 'name', 'System')}]: {latest_msg.content}\n")

[None]: Process dossier at data/test_dossiers/ID-9982_UT_Scan.pdf. Extract thickness grid, run ASME Section VIII Div 1 calculations, check CVC compliance, and publish signed NFA.



2026-09-21 14:49:24,225 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[supervisor]: {"next": "vision_agent", "instruction": "Extract thickness grid from PDF"}



2026-09-21 14:49:26,722 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 14:49:28,367 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 14:49:30,242 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 14:49:30,291 - tools.file_io - INFO - Executing tool: read_scanned_pdf



--- EXECUTING TOOL: read_scanned_pdf ---



...

[reasoning_agent]: 
Okay, I can perform the RBI analysis for you. Please provide the following parameters:

1. **Remaining Life Estimate**: The corrected remaining life estimate (if available). Since the previous value was negative, please clarify the actual remaining service life or the time left before failure.
2. **Fluid Toxicity**: The toxicity level of the contained fluid (e.g., low, moderate, high, severe).
3. **Toxicity Factor**: A factor derived from the fluid toxicity (e.g., based on industry standards or guidelines).
4. **Pressure Factor**: A factor related to the design pressure (e.g., based on stress categories).
5. **Life Factor**: A factor representing the remaining life under inspection conditions.

Once you provide these details, I can proceed with the calculation.



2026-09-21 14:55:30,922 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[supervisor]: {"next": "reasoning_agent", "instruction": "Perform RBI analysis using corrected remaining life estimate and provided parameters"}



2026-09-21 14:55:44,791 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


KeyboardInterrupt: 

In [ ]:
if __name__ == "__main__":
    initial_input = {
        "messages": [
            HumanMessage(
                content="I need you to write a Python script that calculates the volume of a cylindrical vessel with a radius of 1200mm and a height of 5000mm. Execute this script in the secure sandbox and give me the result."
            )
        ],
        "retry_count": 0,
        "max_retries": 3,
        "human_approved": False,
        "human_feedback": ""
    }

    # Stream execution through Ollama LLM nodes
    for chunk in app.stream(initial_input, stream_mode="values"):
        latest_msg = chunk["messages"][-1]
        print(f"[{getattr(latest_msg, 'name', 'System')}]: {latest_msg.content}\n")


[None]: I need you to write a Python script that calculates the volume of a cylindrical vessel with a radius of 1200mm and a height of 5000mm. Execute this script in the secure sandbox and give me the result.



2026-09-21 02:54:00,825 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[supervisor]: {"next": "coder_agent", "instruction": "Calculate the volume of a cylindrical vessel with a radius of 1200mm and a height of 5000mm using the formula V = πr²h"}



2026-09-21 02:54:05,217 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:54:05,268 - tools.docker_sandbox - INFO - Executing tool: execute_in_sandbox



--- EXECUTING TOOL: execute_in_sandbox ---



...

[supervisor]: {"next": "chief_reviewer", "instruction": "Please provide the relevant design specifications and industry standards for the cylindrical vessel, such as ASME Boiler and Pressure Vessel Code, API, or other relevant regulations."}



2026-09-21 02:56:57,921 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[chief_reviewer]: {"name": "provide_design_specs", "parameters": {"design_specs": "The cylindrical vessel should comply with the following design specifications and industry standards:\n\n* ASME Boiler and Pressure Vessel Code (2017 Edition):\n  + Section VIII, Division 1: Pressure Vessels\n  + Section I, Division 1: Power Boilers\n\n* API 650: Tank Fabrication, Welding, and Coating\n\n* Relevant regulatory standards:\n  + OSHA (Occupational Safety and Health Administration) regulations for workplace safety\n  + NFPA (National Fire Protection Association) standards for fire protection and prevention\n"}}



2026-09-21 02:57:14,705 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:57:15,457 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[None]: Based on the provided design specifications and industry standards, the calculated volume of the cylindrical vessel is within the design specifications and complies with the relevant industry standards.

The ASME Boiler and Pressure Vessel Code (2017 Edition) specifies the requirements for pressure vessels and power boilers, which the cylindrical vessel meets. The API 650 standard for tank fabrication, welding, and coating also applies to the cylindrical vessel. Additionally, the OSHA regulations for workplace safety and NFPA standards for fire protection and prevention are relevant to the cylindrical vessel.

Therefore, the calculated volume of the cylindrical vessel is compliant with the design specifications and industry standards.



In [ ]:
if __name__ == "__main__":
    initial_input = {
        "messages": [
            HumanMessage(
                content="""I need you to write a Python script that creates a custom Word Document (.docx) summarizing the health of a pressure vessel. Use the `python-docx` library.

The document should contain:
1. A heading: 'Agentic AI Custom Summary Report'
2. A small table with mock metrics (e.g., Corrosion Pits Found: 42, Remaining Life: 5 Years).

Because you are running in a secure, air-gapped Docker sandbox, YOU MUST NOT SAVE THE FILE TO DISK. 
Instead, you must:
1. Save the document to an in-memory `io.BytesIO` buffer.
2. Encode the buffer to base64.
3. Print the base64 string exactly wrapped inside <<<<FILE_START>>>> and <<<<FILE_END>>>> tags so the host system can extract it.

Execute this script in the secure sandbox."""
            )
        ],
        "retry_count": 0,
        "max_retries": 3,
        "human_approved": False,
        "human_feedback": ""
    }

    # Stream execution through Ollama LLM nodes
    for chunk in app.stream(initial_input, stream_mode="values"):
        latest_msg = chunk["messages"][-1]
        print(f"[{getattr(latest_msg, 'name', 'System')}]: {latest_msg.content}\n")


[None]: I need you to write a Python script that creates a custom Word Document (.docx) summarizing the health of a pressure vessel. Use the `python-docx` library.

The document should contain:
1. A heading: 'Agentic AI Custom Summary Report'
2. A small table with mock metrics (e.g., Corrosion Pits Found: 42, Remaining Life: 5 Years).

Because you are running in a secure, air-gapped Docker sandbox, YOU MUST NOT SAVE THE FILE TO DISK. 
Instead, you must:
1. Save the document to an in-memory `io.BytesIO` buffer.
2. Encode the buffer to base64.
3. Print the base64 string exactly wrapped inside <<<<FILE_START>>>> and <<<<FILE_END>>>> tags so the host system can extract it.

Execute this script in the secure sandbox.



2026-09-21 02:57:21,625 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[supervisor]: ```python
import docx
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from io import BytesIO
import base64

# Create a new document
document = docx.Document()

# Add a heading
heading = document.add_heading('Agentic AI Custom Summary Report', 0)
heading.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add a table
table = document.add_table(rows=2, cols=2, style='Table Grid')
cell = table.cell(0, 0)
cell.text = 'Corrosion Pits Found:'
cell = table.cell(0, 1)
cell.text = '42'

cell = table.cell(1, 0)
cell.text = 'Remaining Life:'
cell = table.cell(1, 1)
cell.text = '5 Years'

# Save the document to an in-memory buffer
buffer = BytesIO()
document.save(buffer)

# Encode the buffer to base64
encoded_buffer = base64.b64encode(buffer.getvalue())

# Print the base64 string
print('<<<<FILE_START>>>>')
print(encoded_buffer.decode('utf-8'))
print('<<<<FILE_END>>>>')
```

This script will create a Word document with the specified heading and table, save it to an in-m

2026-09-21 02:57:40,795 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[chief_reviewer]: Here's the Python script that creates a custom Word Document (.docx) summarizing the health of a pressure vessel using the `python-docx` library:

```python
import docx
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from io import BytesIO
import base64

# Create a new document
document = docx.Document()

# Add a heading
heading = document.add_heading('Agentic AI Custom Summary Report', 0)
heading.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add a table
table = document.add_table(rows=2, cols=2, style='Table Grid')
cell = table.cell(0, 0)
cell.text = 'Corrosion Pits Found:'
cell = table.cell(0, 1)
cell.text = '42'

cell = table.cell(1, 0)
cell.text = 'Remaining Life:'
cell = table.cell(1, 1)
cell.text = '5 Years'

# Save the document to an in-memory buffer
buffer = BytesIO()
document.save(buffer)

# Encode the buffer to base64
encoded_buffer = base64.b64encode(buffer.getvalue())

# Print the base64 string
print('<<<<FILE_START>>>>')
print(encoded_

2026-09-21 02:57:58,637 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:02,534 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:06,387 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:10,550 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:14,804 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:19,122 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:23,569 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:28,108 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 02:58:32,767 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21

[None]: The error message indicates that the `template_type` field is required in the `request` payload. To fix this error, you can add the `template_type` field to the `request` payload with a valid value, such as 'Word'.

Here is the corrected code:

```
{"name": "generate_nfa_documents", "parameters": {"request":{"base_name":"Agentic AI Custom Summary Report","payload":{"document_content":"Agentic AI Custom Summary Report","document_type":"Word","encoding":"base64","file_path":"in-memory buffer","table":[{"Corrosion Pits Found":"42"},{"Remaining Life":"5 Years"}],"template_type":"Word"}}}}
```

This code adds the `template_type` field to the `request` payload with a value of 'Word', which should resolve the error.



# ---------------------------------------------------------
# Direct Tool Testing Sandbox (LLM Driven)
# ---------------------------------------------------------

### LLM Tool Call Test: ASME Calculator

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from tools.asme_calculator import calculate_asme_stresses

print('Testing LLM tool execution for ASME Calculator...')
llm = ChatOllama(model="llama3.1:8b", temperature=0)
llm_with_tools = llm.bind_tools([calculate_asme_stresses])

msg = HumanMessage(content="Calculate the ASME Section VIII minimum thickness for equipment TEST-1 with design pressure 1.5 MPa, radius 1200mm, allowable stress 137.9 MPa, joint efficiency 1.0, CA 3.0mm, measured thickness 16.5mm, and corrosion rate 0.15mm/yr.")
result = llm_with_tools.invoke([msg])

if result.tool_calls:
    print(f"✅ Success! LLM generated tool call: {result.tool_calls[0]['name']}")
    print(f"Arguments: {result.tool_calls[0]['args']}")
    
    print("\n--- Executing Tool with LLM Arguments ---")
    tool_args = result.tool_calls[0]['args']
    execution_result = calculate_asme_stresses.invoke(tool_args)
    print(f"Tool Output: {execution_result}")
else:
    print(f"❌ Failed! LLM generated text instead: {result.content}")

Testing LLM tool execution for ASME Calculator...


2026-09-21 03:00:10,908 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 03:00:10,954 - tools.asme_calculator - INFO - Executing tool: calculate_asme_stresses


✅ Success! LLM generated tool call: calculate_asme_stresses
Arguments: {'corrosion_allowance_mm': 3, 'measured_thickness_mm': 16.5, 'corrosion_rate_mm_yr': 0.15, 'equipment_id': 'TEST-1', 'design_pressure_mpa': 1.5, 'inside_radius_mm': 1200, 'allowable_stress_mpa': 137.9, 'joint_efficiency': 1}

--- Executing Tool with LLM Arguments ---

--- EXECUTING TOOL: calculate_asme_stresses ---

Tool Output: {'t_req_mm': 16.14, 'delta_mm': 0.36, 'is_breach': False, 'status': 'SAFE', 'remaining_life_years': 2.41, 'derated_mawp_mpa': 1.54}


### LLM Tool Call Test: Docker Sandbox

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from tools.docker_sandbox import execute_in_sandbox

print('Testing LLM tool execution for Docker Sandbox...')
llm = ChatOllama(model="llama3.1:8b", temperature=0)
llm_with_tools = llm.bind_tools([execute_in_sandbox])

msg = HumanMessage(content="Write a python script that prints 'Hello Sandbox' and execute it in the secure sandbox.")
result = llm_with_tools.invoke([msg])

if result.tool_calls:
    print(f"✅ Success! LLM generated tool call: {result.tool_calls[0]['name']}")
    print(f"Arguments: {result.tool_calls[0]['args']}")
    
    print("\n--- Executing Tool with LLM Arguments ---")
    tool_args = result.tool_calls[0]['args']
    execution_result = execute_in_sandbox.invoke(tool_args)
    print(f"Tool Output: {execution_result}")
else:
    print(f"❌ Failed! LLM generated text instead: {result.content}")

Testing LLM tool execution for Docker Sandbox...


2026-09-21 03:00:12,025 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 03:00:12,070 - tools.docker_sandbox - INFO - Executing tool: execute_in_sandbox


✅ Success! LLM generated tool call: execute_in_sandbox
Arguments: {'code_string': 'print('}

--- Executing Tool with LLM Arguments ---

--- EXECUTING TOOL: execute_in_sandbox ---

Tool Output: {'status': 'BLOCKED_BY_AST', 'output': "Syntax Error in AI generated code: '(' was never closed (<unknown>, line 1)", 'error_details': None}


### LLM Tool Call Test: API 579 FFS

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from tools.api_579_ffs_tool import run_ffs_assessment

print('Testing LLM tool execution for API 579 FFS Tool...')
llm = ChatOllama(model="llama3.1:8b", temperature=0)
llm_with_tools = llm.bind_tools([run_ffs_assessment])

msg = HumanMessage(content="Run an API 579 FFS Assessment for a vessel with global actual thickness 16.5mm, LTA min thickness 12.0mm, required thickness 16.0mm, flaw length 100mm, inside radius 1200mm, and allowable RSF 0.90.")
result = llm_with_tools.invoke([msg])

if result.tool_calls:
    print(f"✅ Success! LLM generated tool call: {result.tool_calls[0]['name']}")
    print(f"Arguments: {result.tool_calls[0]['args']}")
    
    print("\n--- Executing Tool with LLM Arguments ---")
    tool_args = result.tool_calls[0]['args']
    execution_result = run_ffs_assessment.invoke(tool_args)
    print(f"Tool Output: {execution_result}")
else:
    print(f"❌ Failed! LLM generated text instead: {result.content}")

Testing LLM tool execution for API 579 FFS Tool...


2026-09-21 03:00:15,768 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 03:00:15,816 - tools.api_579_ffs_tool - INFO - Executing tool: run_ffs_assessment


✅ Success! LLM generated tool call: run_ffs_assessment
Arguments: {'t_actual_global': 16.5, 't_min_lta': 12, 't_req': 16, 'flaw_length_mm': 100, 'inside_radius_mm': 1200, 'allowable_rsf': 0.9}

--- Executing Tool with LLM Arguments ---

--- EXECUTING TOOL: run_ffs_assessment ---

Tool Output: {'status': 'SUCCESS', 'error': None, 'rsf': 0.971, 'allowable_rsf': 0.9, 'is_acceptable': True, 'action': 'Acceptable for continued operation based on Level 1 RSF assessment.'}


### LLM Tool Call Test: Compliance Auditor

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from tools.compliance_auditor import audit_cvc_compliance

print('Testing LLM tool execution for Compliance Auditor...')
llm = ChatOllama(model="llama3.1:8b", temperature=0)
llm_with_tools = llm.bind_tools([audit_cvc_compliance])

msg = HumanMessage(content="Audit CVC compliance for procurement request REQ-123. It is a single source procurement for 500000 INR. It has no PAC but it is an emergency, justified by 'CVC Circular 02/02/2004 Emergency Exception', approved by the Director.")
result = llm_with_tools.invoke([msg])

if result.tool_calls:
    print(f"✅ Success! LLM generated tool call: {result.tool_calls[0]['name']}")
    print(f"Arguments: {result.tool_calls[0]['args']}")
    
    print("\n--- Executing Tool with LLM Arguments ---")
    tool_args = result.tool_calls[0]['args']
    execution_result = audit_cvc_compliance.invoke(tool_args)
    print(f"Tool Output: {execution_result}")
else:
    print(f"❌ Failed! LLM generated text instead: {result.content}")

Testing LLM tool execution for Compliance Auditor...


2026-09-21 03:00:19,747 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 03:00:19,793 - tools.compliance_auditor - INFO - Executing tool: audit_cvc_compliance
2026-09-21 03:00:19,794 - tools.compliance_auditor - ERROR - Error in audit_cvc_compliance: 3 validation errors for ProcurementRequest
equipment_id
  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': 'Director'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
estimated_cost_lakhs
  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': 'Director'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
required_financial_authority
  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': 'Director'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


✅ Success! LLM generated tool call: audit_cvc_compliance
Arguments: {'has_pac': False, 'is_emergency': True, 'applicable_cvc_clause': 'CVC Circular 02/02/2004 Emergency Exception', 'dop_authority': 'Director', 'request_id': 'REQ-123', 'amount_inr': 500000, 'is_single_source': True}

--- Executing Tool with LLM Arguments ---

--- EXECUTING TOOL: audit_cvc_compliance ---

Tool Output: {'status': 'error', 'error': "3 validation errors for ProcurementRequest\nequipment_id\n  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': 'Director'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nestimated_cost_lakhs\n  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': 'Director'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nrequired_financial_authority\n  Field required [type=missing, input_value={'request_id': 'REQ-123',..._authority': '

## 🛡️ Audit Ledger Dashboard
Run the cell below to securely query the cryptographic audit ledger and view the latest tool executions, inputs, and errors across all agents.

In [ ]:
import sqlite3
import pandas as pd
import os

# Find the DB depending on which folder the notebook is in
db_path = '../data/audit/audit_ledger.db'
if not os.path.exists(db_path):
    db_path = 'data/audit/audit_ledger.db'

def show_audit_ledger():
    try:
        conn = sqlite3.connect(db_path)
        # Removed the 'LIMIT 15' from the SQL query to get all rows
        df = pd.read_sql_query("SELECT id, timestamp, workflow_id, tool_name, caller, status FROM audit_chain ORDER BY id DESC", conn)
        conn.close()
        
        # Tell pandas to display all rows without truncating
        pd.set_option('display.max_rows', None)
        
        # Style the dataframe for the notebook
        def color_status(val):
            color = 'red' if 'ERROR' in str(val) or 'FAILED' in str(val) or 'BLOCKED' in str(val) else 'green'
            return f'color: {color}; font-weight: bold'
            
        display(df.style.map(color_status, subset=['status']))
        
        # Reset the pandas option back to default if you don't want it affecting other cells
        pd.reset_option('display.max_rows')
        
    except Exception as e:
        print(f"Ledger not found or empty: {e}")

show_audit_ledger()


,id,timestamp,workflow_id,tool_name,caller,status
0,3,2026-09-21T09:13:18.293264+00:00,ASME_UG27_AUDIT,asme_calculator,coder_agent,COMPLETED_BREACH
1,2,2026-09-21T09:10:47.097939+00:00,WORKSPACE_FILE_IO,file_io,coder_agent,FAILED
2,1,2026-09-21T09:10:47.073028+00:00,INIT,system_init,system_admin,INITIALIZED


In [16]:
if __name__ == "__main__":
    initial_input = {
        "messages": [
            HumanMessage(
                content=(
                    "I need you to write a Python script using the python-docx library. "
                    "The script should use input() to ask the user for values like design pressure, "
                    "inside radius, allowable stress, joint efficiency, and measured thickness. "
                    "It should then perform ASME Section VIII Div 1 (UG-27) calculations and "
                    "generate a custom Word document summarizing the inputs and results in a table. "
                    "CRITICAL: Do NOT execute this in the sandbox! The interactive input() prompts "
                    "will cause the sandbox to freeze. Just provide the final python code to me."
                )
            )
        ],
        "retry_count": 0,
        "max_retries": 3,
        "human_approved": False,
        "human_feedback": ""
    }

    # Stream execution through your AI orchestrator
    print("Sending task to Supervisor Agent...\n")
    for chunk in app.stream(initial_input, stream_mode="values"):
        latest_msg = chunk["messages"][-1]
        print(f"[{getattr(latest_msg, 'name', 'System')}]: {latest_msg.content}\n")


Sending task to Supervisor Agent...

[None]: I need you to write a Python script using the python-docx library. The script should use input() to ask the user for values like design pressure, inside radius, allowable stress, joint efficiency, and measured thickness. It should then perform ASME Section VIII Div 1 (UG-27) calculations and generate a custom Word document summarizing the inputs and results in a table. CRITICAL: Do NOT execute this in the sandbox! The interactive input() prompts will cause the sandbox to freeze. Just provide the final python code to me.



2026-09-21 14:56:39,332 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Failed to parse Supervisor JSON: Expecting value: line 1 column 1 (char 0)
[supervisor]: ```python
import docx
import math

def calculate_stress(design_pressure, inside_radius, allowable_stress, joint_efficiency, measured_thickness):
    # UG-27 calculations
    allowable_stress = allowable_stress * joint_efficiency
    design_stress = design_pressure * inside_radius / (measured_thickness * 1.0)
    if design_stress <= allowable_stress:
        result = "Acceptable"
    else:
        result = "Not Acceptable"
    return result

def generate_word_document(design_pressure, inside_radius, allowable_stress, joint_efficiency, measured_thickness, result):
    # Create a new Word document
    doc = docx.Document()

    # Add a title
    doc.add_heading('ASME Section VIII Div 1 (UG-27) Calculations', 0)

    # Add a table with the inputs and results
    table = doc.add_table(rows=6, cols=2)
    table.cell(0, 0).text = 'Design Pressure (psi)'
    table.cell(0, 1).text = str(design_pressure)
   

2026-09-21 14:57:12,335 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[chief_reviewer]: ```python
import docx
import math

def calculate_stress(design_pressure, inside_radius, allowable_stress, joint_efficiency, measured_thickness):
    """
    Perform ASME Section VIII Div 1 (UG-27) calculations.

    Args:
        design_pressure (float): Design pressure (psi)
        inside_radius (float): Inside radius (in)
        allowable_stress (float): Allowable stress (psi)
        joint_efficiency (float): Joint efficiency
        measured_thickness (float): Measured thickness (in)

    Returns:
        str: Result of the calculation ("Acceptable" or "Not Acceptable")
    """
    # UG-27 calculations
    allowable_stress = allowable_stress * joint_efficiency
    design_stress = design_pressure * inside_radius / (measured_thickness * 1.0)
    if design_stress <= allowable_stress:
        result = "Acceptable"
    else:
        result = "Not Acceptable"
    return result

def generate_word_document(design_pressure, inside_radius, allowable_stress, joint_efficien

2026-09-21 14:57:37,448 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-09-21 14:57:39,267 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[None]: Here is the corrected Python script:

```python
import docx
import math

def calculate_stress(design_pressure, inside_radius, allowable_stress, joint_efficiency, measured_thickness):
    """
    Perform ASME Section VIII Div 1 (UG-27) calculations.

    Args:
        design_pressure (float): Design pressure (psi)
        inside_radius (float): Inside radius (in)
        allowable_stress (float): Allowable stress (psi)
        joint_efficiency (float): Joint efficiency
        measured_thickness (float): Measured thickness (in)

    Returns:
        str: Result of the calculation ("Acceptable" or "Not Acceptable")
    """
    # UG-27 calculations
    allowable_stress = allowable_stress * joint_efficiency
    design_stress = design_pressure * inside_radius / (measured_thickness * 1.0)
    if design_stress <= allowable_stress:
        result = "Acceptable"
    else:
        result = "Not Acceptable"
    return result

def generate_word_document(design_pressure, inside_radius, allo